---

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppestana-supervisor/PDI_python/blob/main/topico3/Topico3.ipynb)

> **Nota importante**
>
> Este notebook é carregado diretamente a partir do repositório oficial no GitHub e é apenas para **leitura**.  
> As alterações **não ficam guardadas**.
>
> Para guardar uma versão pessoal:
> - No Colab, ir a **Ficheiro → Guardar uma cópia no Drive**.  
> - A cópia ficará disponível em `O meu Drive / Colab Notebooks`.
>
> Para colaboração:
> - Abrir a versão pessoal no Google Drive.  
> - Clicar em **Partilhar** → **Qualquer pessoa com o link pode editar** (ou adicionar os colegas manualmente).  
> - A partir daí, todos podem editar em tempo real, como num Google Docs.
>
> Para guardar resultados (imagens, ficheiros, etc.) no  Drive:
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> ```
> Depois gravar, por exemplo, em:
> ```python
> img.save('/content/drive/MyDrive/PDI_outputs/resultado.png')
> ```

---

In [ ]:
### !!! Executar esta célula primeiro, se dentro do Colab
# Quando dentro do Colab, clonar o repositório completo para garantir acesso à pasta de imagens
!git clone https://github.com/ppestana-supervisor/PDI_python.git
%cd PDI_python/topico5


# 7. Viés e Ética em Processamento de Imagem

Este notebook explora como **dados desbalanceados podem introduzir viés** em modelos de classificação de imagem. Também abordamos formas de **detectar e mitigar esses enviesamentos**, com sugestões para desenvolvimento futuro.



## 7.1. Simulação de viés no dataset CIFAR-10

Este notebook introduz um tema essencial da aprendizagem automática aplicada — **o viés (bias) nos dados** — mostrando como uma distribuição desigual de classes pode afetar o comportamento e a justiça de um modelo.

1. **Transformações básicas:**  
   Cada imagem do *CIFAR-10* é normalizada para o intervalo \([-1, 1]\) com `transforms.Normalize((0.5,), (0.5,))`.  
   Não há *data augmentation*, pois o objetivo é observar o impacto da distribuição, não da variabilidade.

2. **Carregamento do conjunto CIFAR-10:**  
   O *dataset* contém 60 000 imagens (32×32 px) em 10 classes (`airplane`, `automobile`, `bird`, `cat`, …, `truck`).  
   Aqui usa-se a parte de treino (50 000 imagens).

3. **Simulação de viés:**  
   - Seleciona-se a classe com índice `3` (gatos).  
   - Reduz-se artificialmente essa classe para apenas **100 amostras**.  
   - As restantes classes mantêm-se completas.  
   O resultado é um *dataset enviesado*, onde a classe “cat” representa uma fração muito menor do que as outras.

4. **Construção do novo conjunto:**  
   O `Subset` do PyTorch é usado para criar um subconjunto do *dataset* original com índices personalizados (`biased_indices`).  
   Este é um padrão comum em experiências controladas sobre justiça e desbalanceamento.

5. **Inspeção da distribuição:**  
   O código imprime o número de amostras por classe, evidenciando o desbalanceamento.  
   Isso permitirá, nos blocos seguintes, observar o impacto do viés no treino e na performance do modelo.

**Nota pedagógica:**  
Esta célula ilustra um ponto crítico para qualquer engenheiro de IA:  
> “A qualidade e a justiça de um modelo dependem diretamente da representatividade dos dados.”  

Ao simular viés, torna-se possível discutir:
- a importância da amostragem equilibrada;  
- o papel das métricas adequadas (accuracy vs. recall, F1, etc.);  
- e estratégias de mitigação (re-amostragem, *class weighting*, *data synthesis*).  

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Subset
import torchvision.models as models
from torchvision import datasets, transforms
from sklearn.metrics import classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import random

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
class_names = dataset.classes

# Simular viés: manter só 100 imagens da classe 3 (cat), o resto fica completo
target_class = 3
small_subset_idx = [i for i, (_, y) in enumerate(dataset) if y == target_class][:100]
other_classes_idx = [i for i, (_, y) in enumerate(dataset) if y != target_class]
biased_indices = small_subset_idx + other_classes_idx
biased_dataset = Subset(dataset, biased_indices)

print(f"Tamanho total do dataset enviesado: {len(biased_dataset)}")
print("Distribuição de classes (esperada: classe 3 muito reduzida):")
from collections import Counter
labels = [dataset[i][1] for i in biased_indices]
counts = Counter(labels)
for i in range(10):
    print(f"{class_names[i]}: {counts[i]}")


Tamanho total do dataset enviesado: 45100
Distribuição de classes (esperada: classe 3 muito reduzida):
airplane: 5000
automobile: 5000
bird: 5000
cat: 100
deer: 5000
dog: 5000
frog: 5000
horse: 5000
ship: 5000
truck: 5000


## 7.2. Treinar modelo simples

Atenção!!! Horrorosamente lento no CPU. Idealmente correr em GPU.

Esta célula treina um modelo de classificação simples (ResNet-18) sobre o conjunto *CIFAR-10* **com viés artificial** introduzido anteriormente.  
O objetivo não é atingir alta performance, mas observar os efeitos do desbalanceamento nas métricas.

1. **Carregamento dos dados:**  
   O `DataLoader` cria lotes (*batches*) de 64 imagens, baralhados a cada época (`shuffle=True`).  
   O treino é conduzido apenas sobre o *biased_dataset*, onde a classe “cat” tem muito menos exemplos.

2. **Modelo utilizado:**  
   - `ResNet-18` é uma arquitetura convolucional clássica e leve.  
   - É inicializada **sem pesos pré-treinados** (`weights=None`) para tornar o efeito dos dados mais explícito.  
   - A camada final (`model.fc`) é substituída por uma *fully connected* com 10 saídas, correspondentes às classes do *CIFAR-10*.

3. **Função de perda e otimizador:**  
   - A função `CrossEntropyLoss` mede a discrepância entre as probabilidades previstas e as etiquetas verdadeiras.  
   - O otimizador `Adam` é usado com uma taxa de aprendizagem padrão de `0.001`.

4. **Laço de treino com barra de progresso (`tqdm`):**  
   - Cada iteração processa um *batch*, calcula a perda (`loss`), faz *backpropagation* e atualiza os pesos.  
   - A barra mostra o progresso e a perda atual em tempo real.  
   - No final de cada época imprime-se a perda total acumulada.

5. **Objetivo pedagógico:**  
   Treinar num conjunto enviesado permite observar:
   - que o modelo tende a **sub-representar a classe minoritária** (no caso, “cat”);  
   - que a *perda média* pode parecer estável mesmo quando o desempenho é injusto entre classes;  
   - e que métricas simples como *accuracy* podem mascarar desigualdades.

Nos blocos seguintes, a avaliação quantitativa mostrará este desequilíbrio através de métricas como *precision*, *recall* e *F1-score* — essenciais para analisar imparcialidade em modelos supervisionados.

In [2]:
train_loader = DataLoader(biased_dataset, batch_size=64, shuffle=True)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(3):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Época {epoch+1}", leave=True)
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())

    print(f"Época {epoch+1} terminada: perda total = {total_loss:.2f}")

Época 1: 100%|█████████████████████| 705/705 [28:08<00:00,  2.40s/it, loss=1.27]


Época 1 terminada: perda total = 894.86


Época 2: 100%|████████████████████| 705/705 [07:37<00:00,  1.54it/s, loss=0.649]


Época 2 terminada: perda total = 618.10


Época 3: 100%|████████████████████| 705/705 [08:47<00:00,  1.34it/s, loss=0.893]

Época 3 terminada: perda total = 498.32


## 7.3. Avaliação por classe

Esta célula avalia o modelo treinado, apresentando métricas detalhadas para cada classe — um passo crucial quando há **falta de equilibrio entre classes** no conjunto de dados.

1. **Modo de avaliação:**  
   - `model.eval()` desativa camadas como *dropout* e *batch normalization* em modo de treino, garantindo previsões consistentes.  
   - A inferência é feita dentro de um bloco `torch.no_grad()` para poupar memória e acelerar o cálculo.

2. **Coleta de previsões e rótulos reais:**  
   - Para cada *batch*, o modelo gera uma previsão (`outputs`).  
   - `torch.argmax(outputs, dim=1)` devolve o índice da classe com maior probabilidade.  
   - As listas `true_labels` e `pred_labels` acumulam as etiquetas verdadeiras e as previstas, respetivamente.

3. **Cálculo de métricas desagregadas:**  
   - `classification_report` da *scikit-learn* fornece, para cada classe:
     - **Precision (precisão):** proporção de previsões corretas entre as amostras rotuladas como essa classe.  
     - **Recall (revocação):** proporção de amostras dessa classe que o modelo identificou corretamente.  
     - **F1-score:** média harmónica entre precisão e revocação, útil para conjuntos desbalanceados.  
     - **Support:** número de exemplos reais de cada classe.

4. **Leitura dos resultados:**  
   - A classe “cat” (reduzida a apenas 100 amostras) deverá ter *recall* e *F1-score* significativamente mais baixos do que as outras.  
   - Já as classes mais frequentes exibem métricas inflacionadas, dominando o *accuracy* global.

**Nota pedagógica:**  
Este exemplo ilustra a diferença entre *bom desempenho médio* e *desempenho justo*.  
Mesmo com uma *accuracy* elevada, um modelo enviesado pode falhar sistematicamente em classes minoritárias, refletindo **injustiça algorítmica**.  
Por isso, a análise de métricas desagregadas é uma prática obrigatória em qualquer projeto de *Machine Learning responsável*.  

In [3]:
model.eval()
true_labels = []
pred_labels = []

with torch.no_grad():
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        pred_labels.extend(preds)
        true_labels.extend(labels.numpy())
        
print(classification_report(true_labels, pred_labels, target_names=class_names))


              precision    recall  f1-score   support

    airplane       0.84      0.79      0.81      5000
  automobile       0.88      0.87      0.88      5000
        bird       0.68      0.70      0.69      5000
         cat       0.00      0.00      0.00       100
        deer       0.77      0.72      0.75      5000
         dog       0.80      0.67      0.73      5000
        frog       0.67      0.93      0.78      5000
       horse       0.85      0.81      0.83      5000
        ship       0.89      0.90      0.89      5000
       truck       0.87      0.81      0.84      5000

    accuracy                           0.80     45100
   macro avg       0.73      0.72      0.72     45100
weighted avg       0.80      0.80      0.80     45100



/Users/pedropestana/anaconda3/envs/ml_imagem/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/pedropestana/anaconda3/envs/ml_imagem/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/pedropestana/anaconda3/envs/ml_imagem/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

## 7.4. [Future Work] - Fairness em Classificação Visual
*(Possível extensão neste notebook ou ponto de partida para projeto dos estudantes)*

Este notebook pode ser expandido para estudar **fairness** em classificadores de imagem, treinando variantes com dados balanceados e desbalanceados. Algumas possíveis extensões:
- Introduzir métricas como *Demographic Parity* ou *Equal Opportunity*;
- Aplicar *data augmentation* seletivo para minorias;
- Explorar reweighting ou *oversampling* de classes raras;
- Testar impacto em datasets reais com viés conhecido (e.g., UTKFace, FairFace).

Este tema pode ser desenvolvido como projeto de dissertação ou artigo aplicado.

## 7.5. [Future Work] — Robustez de Detetores de Deepfakes a Novos Modelos de Geração
*(Possível extensão neste notebook ou ponto de partida para projeto dos estudantes)*

Esta proposta investiga a **capacidade de generalização dos detetores de deepfakes**,  
testando se um modelo treinado para distinguir imagens reais de um certo gerador (ex: Stable Diffusion v1.5)  
mantém desempenho aceitável quando confrontado com imagens de outros modelos (ex: SDXL, Midjourney, DALL·E 3).

**Hipótese:** os detetores atuais são fortemente dependentes das características específicas do modelo de origem,  
e o seu desempenho cai acentuadamente em *cross-generation evaluation*.

**Pipeline de teste sugerido:**
1. **Dataset:**
   - Coletar ~500 imagens reais (ex: CIFAR-10 ou CelebA);  
   - Gerar 500 imagens falsas com *modelo A* (ex: SD v1.5);  
   - Gerar 500 imagens falsas com *modelo B* (ex: SDXL ou Midjourney).  

2. **Treino:**
   - Treinar um classificador CNN simples (ex: ResNet18) para distinguir *reais vs. falsas (modelo A)*.  
   - Validar desempenho no mesmo domínio (modelo A).

3. **Teste cruzado:**
   - Avaliar o mesmo detetor nas imagens falsas do *modelo B*.  
   - Registar a variação da precisão (*drop rate*) entre domínios.

4. **Análise:**
   - Visualizar *GradCAMs* para verificar se o modelo aprende artefactos superficiais (textura, ruído, compressão).  
   - Correlacionar padrões visuais com falhas de generalização.

5. **Discussão:**
   - Refletir sobre o que constitui “realismo” aprendido pela rede;  
   - Considerar implicações para deteção forense e regulação de IA generativa.

**Extensões possíveis:**
- Repetir o experimento com áudio ou vídeo (deepfake multimodal);
- Introduzir *adversarial fine-tuning* para melhorar a robustez do detetor.

Este estudo integra-se diretamente na vertente **ética e interpretativa da IA visual**,  
demonstrando empiricamente os limites atuais da detecção automática de manipulações digitais.

## 7.6. Discussão prática

- Como garantir representatividade nos dados?
- Como definir limites para o uso de modelos gerativos?
- Como documentar e comunicar as limitações de um modelo?
- Que tipos de auditoria ou validação ética são possíveis?